<a href="https://colab.research.google.com/github/1BF24CS033/BIS/blob/main/Grey_Wolf_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:

import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

np.random.seed(42)

n = 250

cash_flow = np.random.randn(n)
working_capital = np.random.randn(n)
sales_growth = np.random.randn(n) * 0.2
company_size = np.random.randn(n)
leverage = np.random.randn(n)
market_return = np.random.randn(n)

future_cash_flow = (
    2.0
    + 1.5 * cash_flow
    - 0.8 * working_capital
    + 1.2 * sales_growth
    + 0.7 * (sales_growth ** 2)
    + 1.0 * working_capital * (sales_growth ** 2)
    + 0.5 * company_size
    - 0.4 * leverage
    + 0.6 * market_return
    + np.random.randn(n) * 0.2
)

parameter_names = [
    "Intercept",
    "Cash Flow",
    "Working Capital",
    "Sales Growth",
    "Sales Growth Squared",
    "Working Capital x Sales Growth Squared",
    "Company Size",
    "Leverage",
    "Market Return"
]

X = np.column_stack([
    np.ones(n),
    cash_flow,
    working_capital,
    sales_growth,
    sales_growth ** 2,
    working_capital * (sales_growth ** 2),
    company_size,
    leverage,
    market_return
])

y = future_cash_flow

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

def fitness_function(parameters):
    predictions = X_train @ parameters
    mse = np.mean((y_train - predictions) ** 2)
    return mse


def GWO(objective, dimensions, wolves=30, iterations=100):

    positions = np.random.uniform(
        -5, 5, (wolves, dimensions)
    )

    alpha = np.zeros(dimensions)
    beta = np.zeros(dimensions)
    delta = np.zeros(dimensions)

    alpha_score = float("inf")
    beta_score = float("inf")
    delta_score = float("inf")

    convergence = []

    for iteration in range(iterations):

        for i in range(wolves):

            positions[i] = np.clip(positions[i], -5, 5)

            score = objective(positions[i])

            if score < alpha_score:
                delta_score = beta_score
                delta = beta.copy()

                beta_score = alpha_score
                beta = alpha.copy()

                alpha_score = score
                alpha = positions[i].copy()

            elif score < beta_score:
                delta_score = beta_score
                delta = beta.copy()

                beta_score = score
                beta = positions[i].copy()

            elif score < delta_score:
                delta_score = score
                delta = positions[i].copy()

        a = 2 - iteration * (2 / iterations)

        for i in range(wolves):

            for j in range(dimensions):

                r1, r2 = np.random.rand(), np.random.rand()
                A1 = 2 * a * r1 - a
                C1 = 2 * r2

                D_alpha = abs(C1 * alpha[j] - positions[i, j])
                X1 = alpha[j] - A1 * D_alpha

                r1, r2 = np.random.rand(), np.random.rand()
                A2 = 2 * a * r1 - a
                C2 = 2 * r2

                D_beta = abs(C2 * beta[j] - positions[i, j])
                X2 = beta[j] - A2 * D_beta

                r1, r2 = np.random.rand(), np.random.rand()
                A3 = 2 * a * r1 - a
                C3 = 2 * r2

                D_delta = abs(C3 * delta[j] - positions[i, j])
                X3 = delta[j] - A3 * D_delta

                positions[i, j] = (X1 + X2 + X3) / 3



        convergence.append(alpha_score)


        if iteration < 5 or (iteration + 1) % 5 == 0:
            print(
                f"Iteration {iteration + 1}: "
                f"Best MSE = {alpha_score:.6f}"
            )


    return alpha, alpha_score, convergence



best_parameters, final_alpha_score, convergence = GWO(
    objective=fitness_function,
    dimensions=X_train.shape[1],
    wolves=30,
    iterations=100
)


print("\n========== FINAL ALPHA WOLF ==========")

print("Final Alpha Fitness (MSE):",
      f"{final_alpha_score:.6f}")

print("\nFinal Alpha Position:")

for name, value in zip(parameter_names, best_parameters):
    print(f"{name}: {value:.6f}")


predictions = X_test @ best_parameters

mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("\n========== MODEL PERFORMANCE ==========")

print(f"Test Mean Squared Error: {mse:.6f}")
print(f"Test R2 Score: {r2:.6f}")


Iteration 1: Best MSE = 12.668799
Iteration 2: Best MSE = 7.599846
Iteration 3: Best MSE = 2.975659
Iteration 4: Best MSE = 2.176816
Iteration 5: Best MSE = 2.040424
Iteration 10: Best MSE = 0.245045
Iteration 15: Best MSE = 0.215295
Iteration 20: Best MSE = 0.215295
Iteration 25: Best MSE = 0.189776
Iteration 30: Best MSE = 0.141976
Iteration 35: Best MSE = 0.134707
Iteration 40: Best MSE = 0.134707
Iteration 45: Best MSE = 0.132097
Iteration 50: Best MSE = 0.119747
Iteration 55: Best MSE = 0.114032
Iteration 60: Best MSE = 0.100376
Iteration 65: Best MSE = 0.079956
Iteration 70: Best MSE = 0.072541
Iteration 75: Best MSE = 0.070771
Iteration 80: Best MSE = 0.064324
Iteration 85: Best MSE = 0.062142
Iteration 90: Best MSE = 0.059648
Iteration 95: Best MSE = 0.056077
Iteration 100: Best MSE = 0.054377

========== FINAL ALPHA WOLF ==========
Final Alpha Fitness (MSE): 0.054377

Final Alpha Position:
Intercept: 1.973290
Cash Flow: 1.521813
Working Capital: -0.700258
Sales Growth: 0.70816